# STOXX EUROPE 600 : composites factoriels et analyse incrémentale fondés sur les preuves par variable

Ce notebook construit uniquement un pipeline reproductible et n'est pas exécuté lors de sa création. Pour chaque famille, il teste trois sélections à poids égaux : un noyau stable, une confirmation récente et la sélection précédente conservée comme contrôle. Chaque composite est comparé au score de la famille déjà présent dans le screen, puis chaque composante est testée marginalement par rapport à ce facteur existant avec un poids de 25 %.

Tous les résultats seront écrits dans exports/factor_family_pipeline_STOXX600_variants. Pour les périodes courtes, robust_score est utilisé à titre diagnostique et ne doit pas être comparé en niveau aux périodes historiques complètes.

In [ ]:

from pathlib import Path
import json
import pandas as pd
import numpy as np

REPO_ROOT = Path.cwd()
if not (REPO_ROOT / "func.py").exists():
    raise RuntimeError(
        "Définissez le répertoire de travail Jupyter sur C:\\dev\\factor_backtest avant d'exécuter ce notebook."
    )

from func import (
    calculate_benchmark_performance,
    calculate_performance_ratios,
    combine_backtest_performances,
    export_backtest_results,
    load_backtest_data,
    test_composite_signals,
    test_incremental_signals,
)
from factor_config import LOWER_IS_BETTER, signal_options

print("Les fonctions de recherche sont chargées.")


In [ ]:
MARKET = "STOXX EUROPE 600"
BENCHMARK = "STOXX EUROPE 600"
START_DATE = "2007-12-01"
PERCENTILE = 0.13
N_JOBS = 1
INCREMENTAL_BASELINE_WEIGHT = 0.75
INCREMENTAL_CANDIDATE_WEIGHT = 0.25
PERIOD_BREAKPOINTS = [2009, 2013, 2017, 2020, 2022, 2024, 2026]
OUTPUT_NAME = "factor_family_pipeline_STOXX600_variants"
EVIDENCE_REPORT = Path(
    r"C:\dev\factor_backtest\exports\_agent_work\stoxx_regime_fullpool.md"
)

BASELINE_CANDIDATES = {
    "growth": ("GROWTH_SCORE_FS_SECTOR", "Growth Avg Percentile"),
    "quality": ("Quality Avg Percentile", "MARGIN_SCORE_FS_SECTOR"),
    "momentum": ("MOMENTUM_SCORE_FS_SECTOR", "Mom Avg Percentile"),
    "value": ("VALUE_SCORE_FS_SECTOR", "Value Avg Percentile"),
    "dividend": ("Dividend Avg Percentile", "Dividend_NTM Avg Percentile"),
}
SELECTIONS = {
    "dividend": [
        {"role": "long", "variable": "DPS FY1", "dimension": "pct_6", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Croissance du DPS à six mois, persistante sur plusieurs régimes"},
        {"role": "cycle", "variable": "DVD Yield FY0", "dimension": "rank_diff_1", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Amélioration du rang du rendement du dividende"},
        {"role": "short", "variable": "CFO Div Cov Ratio", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Amélioration de la couverture du dividende par le CFO"},
    ],
    "growth": [
        {"role": "long", "variable": "5Y_Hist EPS TrendStab", "dimension": "rank_diff_3", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Stabilité de la tendance historique de l'EPS"},
        {"role": "cycle", "variable": "5Y_Hist GrossInc TrendStab", "dimension": "rank_diff_3", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Stabilité de la tendance du résultat brut"},
        {"role": "short", "variable": "CFO 5Y CAGR", "dimension": "level", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Croissance structurelle des flux de trésorerie"},
    ],
    "momentum": [
        {"role": "long", "variable": "SP Price Close CIQ", "dimension": "pct_12", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Momentum de prix à douze mois"},
        {"role": "cycle", "variable": "SP Price Target CIQ", "dimension": "pct_12", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Variation longue de l'objectif de cours"},
        {"role": "short", "variable": "Mom Avg Percentile", "dimension": "level", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Confirmation par le niveau agrégé de momentum"},
    ],
    "quality": [
        {"role": "long", "variable": "NetDebt to EBITDA exFIN", "dimension": "rank_diff_3", "higher_is_better": False, "evidence_class": "stable_core", "evidence_note": "Amélioration du rang de levier, avec une source lower-is-better"},
        {"role": "cycle", "variable": "Cont Op Earning Margin", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Amélioration trimestrielle de la marge opérationnelle"},
        {"role": "short", "variable": "PCT ROE", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Amélioration du ROE pour diversifier l'exposition au levier"},
    ],
    "value": [
        {"role": "long", "variable": "EV To EBITDA LTM", "dimension": "pct_3", "higher_is_better": False, "evidence_class": "stable_core", "evidence_note": "Baisse du multiple EV sur EBITDA"},
        {"role": "cycle", "variable": "Earns Yield FY0", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Amélioration du rendement bénéficiaire spot"},
        {"role": "short", "variable": "Earns Yield FY1", "dimension": "rank_diff_1", "higher_is_better": True, "evidence_class": "stable_core", "evidence_note": "Amélioration du rang du rendement bénéficiaire forward"},
    ],
}

SELECTIONS_2 = {
    "dividend": [
        {"role": "long", "variable": "DPS FY1", "dimension": "pct_6", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Ancrage DPS complété par des signaux récents"},
        {"role": "cycle", "variable": "DVD Yield FY1", "dimension": "level", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Rendement forward dont l'IR s'améliore sur les périodes récentes"},
        {"role": "short", "variable": "FCF Div Cov Ratio", "dimension": "rank_diff_6", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Confirmation récente par la couverture FCF"},
    ],
    "growth": [
        {"role": "long", "variable": "5Y_Hist GrossInc TrendStab", "dimension": "rank_diff_3", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Ancrage de stabilité du résultat brut"},
        {"role": "cycle", "variable": "CFO 5Y CAGR", "dimension": "diff_1", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Accélération récente de la croissance du CFO"},
        {"role": "short", "variable": "Revenue 5Y CAGR", "dimension": "pct_1", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Accélération récente du chiffre d'affaires"},
    ],
    "momentum": [
        {"role": "long", "variable": "SP Price Close CIQ", "dimension": "pct_12", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Momentum de prix persistant"},
        {"role": "cycle", "variable": "Mom Avg Percentile", "dimension": "level", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Confirmation agrégée dont l'IR récent progresse"},
        {"role": "short", "variable": "PMOM 12M1M", "dimension": "level", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Momentum douze mois hors dernier mois en amélioration récente"},
    ],
    "quality": [
        {"role": "long", "variable": "Cont Op Earning Margin", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Ancrage de rentabilité opérationnelle"},
        {"role": "cycle", "variable": "PCT ROE", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Amélioration de la rentabilité des fonds propres"},
        {"role": "short", "variable": "Total Debt", "dimension": "diff_12", "higher_is_better": False, "evidence_class": "recent_confirmation", "evidence_note": "Baisse de la dette totale, en amélioration entre 2022-2023 et 2024-2025"},
    ],
    "value": [
        {"role": "long", "variable": "EV To EBITDA LTM", "dimension": "pct_3", "higher_is_better": False, "evidence_class": "recent_confirmation", "evidence_note": "Ancrage de valorisation relatif"},
        {"role": "cycle", "variable": "Earns Yield FY1", "dimension": "level", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Rendement bénéficiaire forward en amélioration récente"},
        {"role": "short", "variable": "Earns Yield FY0", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "recent_confirmation", "evidence_note": "Amélioration cyclique du rendement bénéficiaire spot"},
    ],
}

SELECTIONS_3 = {
    "dividend": [
        {"role": "long", "variable": "DPS FY1", "dimension": "pct_6", "higher_is_better": True, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
        {"role": "cycle", "variable": "FCF Div Cov Ratio", "dimension": "rank_diff_6", "higher_is_better": True, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
        {"role": "short", "variable": "CFO Div Cov Ratio", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
    ],
    "growth": [
        {"role": "long", "variable": "5Y_Hist EPS TrendStab", "dimension": "rank_diff_3", "higher_is_better": True, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
        {"role": "cycle", "variable": "CFO 5Y CAGR", "dimension": "level", "higher_is_better": True, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
        {"role": "short", "variable": "Revenue 5Y CAGR", "dimension": "pct_1", "higher_is_better": True, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
    ],
    "momentum": [
        {"role": "long", "variable": "SP Price Target CIQ", "dimension": "pct_12", "higher_is_better": True, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
        {"role": "cycle", "variable": "Pct_Short_Interest", "dimension": "diff_6", "higher_is_better": False, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
        {"role": "short", "variable": "SP Price Close CIQ", "dimension": "rank_diff_1", "higher_is_better": True, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
    ],
    "quality": [
        {"role": "long", "variable": "NetDebt to EBITDA exFIN", "dimension": "rank_diff_3", "higher_is_better": False, "evidence_class": "current_control", "evidence_note": "Sélection précédente après correction de direction"},
        {"role": "cycle", "variable": "Net Debt to Tot Equity", "dimension": "pct_1", "higher_is_better": False, "evidence_class": "current_control", "evidence_note": "Sélection précédente après correction de direction"},
        {"role": "short", "variable": "Cont Op Earning Margin", "dimension": "diff_1", "higher_is_better": True, "evidence_class": "current_control", "evidence_note": "Sélection précédente après correction de direction"},
    ],
    "value": [
        {"role": "long", "variable": "EV To EBITDA LTM", "dimension": "pct_3", "higher_is_better": False, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
        {"role": "cycle", "variable": "Earns Yield FY0", "dimension": "diff_3", "higher_is_better": True, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
        {"role": "short", "variable": "Earns Yield FY1", "dimension": "diff_1", "higher_is_better": True, "evidence_class": "current_control", "evidence_note": "Sélection précédente conservée comme contrôle"},
    ],
}

SELECTION_VARIANTS = {
    "stable_core": SELECTIONS,
    "recent_confirmation": SELECTIONS_2,
    "current_control": SELECTIONS_3,
}


## 1. Sélection des composantes et règle de pondération égale

SELECTIONS contient le noyau stable, SELECTIONS_2 la confirmation récente et SELECTIONS_3 le contrôle précédent. Chaque famille contient trois composantes de poids 1,0. Le score est obtenu par addition des contributions, puis transformé selon la neutralisation cross-sectionnelle déjà utilisée par le pipeline.

In [ ]:

def validate_selection_directions(selections):
    """Vérifie la direction économique de chaque variable brute."""
    errors = []
    for family, specs in selections.items():
        for spec in specs:
            expected_higher = spec["variable"] not in LOWER_IS_BETTER
            if bool(spec["higher_is_better"]) != expected_higher:
                expected_label = "higher" if expected_higher else "lower"
                errors.append(
                    f"{family}: {spec['variable']} doit être {expected_label}-is-better"
                )
    if errors:
        raise ValueError(
            "Directions incompatibles avec factor_config.LOWER_IS_BETTER : "
            + "; ".join(errors)
        )


for variant_name, selections in SELECTION_VARIANTS.items():
    validate_selection_directions(selections)


def make_single_config(spec, weight=1.0):
    kwargs = {"higher_is_better": bool(spec["higher_is_better"])}
    kwargs[spec["dimension"]] = float(weight)
    return {spec["variable"]: signal_options(**kwargs)}


def make_family_config(specs):
    config = {}
    for spec in specs:
        variable = spec["variable"]
        if variable not in config:
            config[variable] = signal_options(
                higher_is_better=bool(spec["higher_is_better"])
            )
        config[variable][f"weight_{spec['dimension']}"] = 1.0
    return config


def make_baseline_config(family, weight=1.0):
    variable = BASELINE_COLUMNS[family]
    return {variable: signal_options(level=float(weight), higher_is_better=True)}


SELECTION_ROWS = []
for variant_name, selections in SELECTION_VARIANTS.items():
    for family, specs in selections.items():
        for index, spec in enumerate(specs, start=1):
            SELECTION_ROWS.append(
                {
                    "market": MARKET,
                    "variant": variant_name,
                    "family": family,
                    "component_index": index,
                    "role": spec["role"],
                    "variable": spec["variable"],
                    "dimension": spec["dimension"],
                    "higher_is_better": spec["higher_is_better"],
                    "evidence_class": spec["evidence_class"],
                    "evidence_note": spec["evidence_note"],
                    "source_report": str(EVIDENCE_REPORT),
                    "composite_weight": 1.0,
                    "baseline_column": None,
                }
            )
SELECTION_MANIFEST = pd.DataFrame(SELECTION_ROWS)
display(SELECTION_MANIFEST)


In [ ]:

DATA_DIR = REPO_ROOT / "data"
SCREEN_PATH = DATA_DIR / "screen_aggregate.parquet"
RETURNS_PATH = DATA_DIR / "returns.parquet"
EXPORT_ROOT = REPO_ROOT / "exports"
EXPORT_DIR = EXPORT_ROOT / OUTPUT_NAME
LIST_NOIRE_PATH = None

try:
    import pyarrow.parquet as pq
    available_columns = set(pq.ParquetFile(SCREEN_PATH).schema_arrow.names)
except Exception as error:
    raise RuntimeError("Échec de lecture du schéma parquet du screen ; vérifiez que pyarrow est disponible.") from error

BASELINE_COLUMNS = {}
for family, candidates in BASELINE_CANDIDATES.items():
    selected = next((candidate for candidate in candidates if candidate in available_columns), None)
    if selected is None:
        raise KeyError(f"Aucune colonne de facteur existante pour {family} dans le screen : {candidates}")
    BASELINE_COLUMNS[family] = selected

if "SELECTION_MANIFEST" in globals():
    SELECTION_MANIFEST["baseline_column"] = SELECTION_MANIFEST["family"].map(BASELINE_COLUMNS)

SELECTED_RAW_VARIABLES = sorted(
    {
        spec["variable"]
        for selections in SELECTION_VARIANTS.values()
        for specs in selections.values()
        for spec in specs
    }
)
LOAD_VARIABLES = list(
    dict.fromkeys(SELECTED_RAW_VARIABLES + list(BASELINE_COLUMNS.values()))
)

screen, returns = load_backtest_data(
    screen_path=SCREEN_PATH,
    returns_path=RETURNS_PATH,
    variables=LOAD_VARIABLES,
    bench=BENCHMARK,
    start_date=START_DATE,
    lookback_periods=12,
    compact_dtypes=True,
)
screen["Date"] = pd.to_datetime(screen["Date"])

missing = [column for column in LOAD_VARIABLES if column not in screen.columns]
if missing:
    raise KeyError(f"Variables absentes après chargement : {missing}")
if f"Weight in {BENCHMARK}" not in screen.columns:
    raise KeyError(f"La colonne Weight in {BENCHMARK} est absente du screen")

BENCH_PERF = calculate_benchmark_performance(
    screen=screen,
    returns=returns,
    bench=BENCHMARK,
    start_date=START_DATE,
)

MONTHLY_BASE_CACHE = {}
RUN_OPTIONS = {
    "bench": BENCHMARK,
    "bench_perf": BENCH_PERF,
    "percentile": PERCENTILE,
    "start_date": START_DATE,
    "freq_rebal": 1,
    "fill_method": "copy",
    "n_jobs": N_JOBS,
    "retain_builders": False,
    "monthly_base_cache": MONTHLY_BASE_CACHE,
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "show_plot": False,
    "build_figure": False,
}

print(f"screen={screen.shape}, returns={returns.shape}")
print(f"benchmark={BENCHMARK}; baseline columns={BASELINE_COLUMNS}")


## 2. Construire et backtester les nouveaux composites familiaux et les facteurs existants du screen

In [ ]:

COMPOSITE_CONFIGS = {}
for variant_name, selections in SELECTION_VARIANTS.items():
    for family, specs in selections.items():
        COMPOSITE_CONFIGS[f"{variant_name}__{family}"] = make_family_config(specs)
for family in SELECTIONS:
    COMPOSITE_CONFIGS[f"screen_baseline_{family}"] = make_baseline_config(family)

composite_batch = test_composite_signals(
    screen=screen,
    returns=returns,
    composite_configs=COMPOSITE_CONFIGS,
    list_noire_path=LIST_NOIRE_PATH,
    score_prefix="Score_FamilyPipeline",
    **RUN_OPTIONS,
)
screen = composite_batch["screen"]
print("Le backtest comparatif des nouveaux composites familiaux et des facteurs existants du screen est terminé.")


## 3. Analyse incrémentale de chaque variable

Chaque batch combine 75 % du facteur de référence de la famille et 25 % d'une seule composante variable × dimension. Les colonnes delta_active_cagr, delta_top_worst_cagr, delta_top_information_ratio, delta_robust_score, delta_active_max_drawdown et delta_tracking_error_annualized permettent de mesurer directement la contribution marginale.

In [ ]:

incremental_batches = {}
for variant_name, selections in SELECTION_VARIANTS.items():
    for family, specs in selections.items():
        for index, spec in enumerate(specs, start=1):
            batch_key = f"{variant_name}__{family}__{spec['role']}__{index}"
            incremental_batches[batch_key] = test_incremental_signals(
                screen=screen,
                returns=returns,
                baseline_config=make_baseline_config(
                    family, weight=INCREMENTAL_BASELINE_WEIGHT,
                ),
                candidate_config=make_single_config(
                    spec, weight=INCREMENTAL_CANDIDATE_WEIGHT,
                ),
                list_noire_path=LIST_NOIRE_PATH,
                **RUN_OPTIONS,
            )
            screen = incremental_batches[batch_key]["screen"]

all_results = {
    "composite_comparison": composite_batch,
    "incremental": incremental_batches,
}
print(f"{len(incremental_batches)} batches incrémentaux à une composante sont terminés.")


## 4. Export normalisé

L'export conserve les métriques officielles, les courbes de performance, le tableau de comparaison composite/facteur existant, le tableau incremental par composante et le manifest de sélection. Il ne faut pas regarder uniquement le CAGR total : la sélection doit combiner les périodes économiques complètes, Top/Worst, IR, drawdown, tracking error et les gates.

In [ ]:

exported = export_backtest_results(
    results=all_results,
    output_dir=EXPORT_ROOT,
    export_name=OUTPUT_NAME,
    export_html=False,
    export_png=False,
    export_holdings=False,
)
EXPORT_DIR = Path(exported["export_dir"])

metrics = pd.read_csv(EXPORT_DIR / "backtest_metrics.csv")
with (EXPORT_DIR / "backtest_registry.json").open("r", encoding="utf-8") as handle:
    registry = json.load(handle)
path_by_name = {
    entry.get("metadata", {}).get("test_name"): entry.get("test_path")
    for entry in registry
    if entry.get("metadata", {}).get("test_name") and entry.get("test_path")
}

METRIC_COLUMNS = [
    "active_cagr",
    "top_worst_cagr",
    "top_information_ratio",
    "robust_score",
    "active_max_drawdown",
    "tracking_error_annualized",
    "min_rolling_3y_cagr",
    "top_bench_ratio",
    "top_worst_ratio",
    "top_annualized_return",
    "bench_annualized_return",
    "observation_count",
    "years",
]
COMPARABILITY_COLUMNS = [
    "robust_score_comparable"
] if "robust_score_comparable" in metrics.columns else []


def _path_for(test_name):
    if test_name not in path_by_name:
        raise KeyError(f"Le test_name={test_name} est absent du registry exporté")
    return path_by_name[test_name]


def _metric_slice(test_path):
    return metrics.loc[metrics["test_path"].eq(test_path)].copy()


family_comparison_parts = []
family_names = list(SELECTIONS)
for variant_name, selections in SELECTION_VARIANTS.items():
    for family in family_names:
        new_rows = _metric_slice(_path_for(f"{variant_name}__{family}"))
        base_rows = _metric_slice(_path_for(f"screen_baseline_{family}"))
        left_columns = ["period_id", "scope", "period_label", *METRIC_COLUMNS, *COMPARABILITY_COLUMNS]
        right_columns = ["period_id", "scope", *METRIC_COLUMNS, *COMPARABILITY_COLUMNS]
        left = new_rows[left_columns].rename(
            columns={column: f"{column}_new" for column in [*METRIC_COLUMNS, *COMPARABILITY_COLUMNS]}
        )
        right = base_rows[right_columns].rename(
            columns={column: f"{column}_screen" for column in [*METRIC_COLUMNS, *COMPARABILITY_COLUMNS]}
        )
        joined = left.merge(right, on=["period_id", "scope"], how="outer")
        joined.insert(0, "family", family)
        joined.insert(0, "variant", variant_name)
        for column in METRIC_COLUMNS:
            joined[f"delta_{column}"] = (
                joined[f"{column}_new"] - joined[f"{column}_screen"]
            )
        joined["new_perf_gate"] = (
            joined["active_cagr_new"].gt(0)
            & joined["top_worst_cagr_new"].gt(0)
            & joined["top_information_ratio_new"].gt(0)
        )
        joined["screen_perf_gate"] = (
            joined["active_cagr_screen"].gt(0)
            & joined["top_worst_cagr_screen"].gt(0)
            & joined["top_information_ratio_screen"].gt(0)
        )
        joined["performance_improved"] = (
            joined["delta_active_cagr"].gt(0)
            & joined["delta_top_worst_cagr"].gt(0)
            & joined["delta_top_information_ratio"].gt(0)
        )
        joined["risk_not_worse"] = (
            joined["delta_active_max_drawdown"].le(0)
            & joined["delta_tracking_error_annualized"].le(0)
        )
        if COMPARABILITY_COLUMNS:
            comparable_period = (
                joined["scope"].eq("total")
                | (
                    joined["robust_score_comparable_new"].astype(str).str.lower().isin(["true", "1", "yes"])
                    & joined["robust_score_comparable_screen"].astype(str).str.lower().isin(["true", "1", "yes"])
                )
            )
        else:
            comparable_period = joined["scope"].eq("total")
        joined["strict_comparable_improvement"] = (
            comparable_period
            & joined["performance_improved"]
            & joined["risk_not_worse"]
            & joined["robust_score_new"].gt(joined["robust_score_screen"])
        )
        family_comparison_parts.append(joined)

family_comparison = pd.concat(family_comparison_parts, ignore_index=True)
family_comparison.to_csv(
    EXPORT_DIR / "family_composite_vs_screen.csv", index=False
)
family_comparison.loc[family_comparison["period_id"].eq("total")].to_csv(
    EXPORT_DIR / "family_composite_vs_screen_total.csv", index=False
)

incremental_lookup = {}
for variant_name, selections in SELECTION_VARIANTS.items():
    for family, specs in selections.items():
        for index, spec in enumerate(specs, start=1):
            incremental_lookup[f"{variant_name}__{family}__{spec['role']}__{index}"] = {
                "variant": variant_name,
                "family": family,
                "role": spec["role"],
                "variable": spec["variable"],
                "dimension": spec["dimension"],
                "higher_is_better": spec["higher_is_better"],
                "baseline_weight": INCREMENTAL_BASELINE_WEIGHT,
                "candidate_weight": INCREMENTAL_CANDIDATE_WEIGHT,
            }

incremental_rows = []
for test_group in metrics.loc[
    metrics["test_type"].isin(
        ["incremental_baseline", "incremental_candidate"]
    ),
    "test_group",
].dropna().unique():
    group_rows = metrics.loc[metrics["test_group"].eq(test_group)].copy()
    baseline_rows = group_rows.loc[
        group_rows["test_type"].eq("incremental_baseline")
    ]
    candidate_rows = group_rows.loc[
        group_rows["test_type"].eq("incremental_candidate")
    ]
    batch_key = str(test_group).split(" / ")[-1]
    spec_info = incremental_lookup.get(batch_key, {})
    for _, candidate in candidate_rows.iterrows():
        baseline = baseline_rows.loc[
            baseline_rows["period_id"].eq(candidate["period_id"])
        ]
        if baseline.empty:
            continue
        baseline = baseline.iloc[0]
        row = {
            "batch_key": batch_key,
            **spec_info,
            "period_id": candidate["period_id"],
            "scope": candidate["scope"],
            "period_label": candidate.get("period_label"),
            "candidate_test_path": candidate["test_path"],
            "baseline_test_path": baseline["test_path"],
        }
        for column in METRIC_COLUMNS:
            row[f"{column}_candidate"] = candidate.get(column)
            row[f"{column}_baseline"] = baseline.get(column)
            row[f"delta_{column}"] = (
                candidate.get(column) - baseline.get(column)
            )
        row["candidate_perf_gate"] = (
            candidate["active_cagr"] > 0
            and candidate["top_worst_cagr"] > 0
            and candidate["top_information_ratio"] > 0
        )
        row["baseline_perf_gate"] = (
            baseline["active_cagr"] > 0
            and baseline["top_worst_cagr"] > 0
            and baseline["top_information_ratio"] > 0
        )
        row["incremental_perf_improved"] = (
            row["delta_active_cagr"] > 0
            and row["delta_top_worst_cagr"] > 0
            and row["delta_top_information_ratio"] > 0
        )
        row["incremental_risk_not_worse"] = (
            row["delta_active_max_drawdown"] <= 0
            and row["delta_tracking_error_annualized"] <= 0
        )
        incremental_rows.append(row)

incremental_effects = pd.DataFrame(incremental_rows)
incremental_effects.to_csv(EXPORT_DIR / "incremental_effects.csv", index=False)
incremental_effects.loc[
    incremental_effects["period_id"].eq("total")
].to_csv(EXPORT_DIR / "incremental_effects_total.csv", index=False)

performance_selections = {}
first_baseline_path = None
for variant_name in SELECTION_VARIANTS:
    for family in family_names:
        new_path = _path_for(f"{variant_name}__{family}")
        performance_selections[f"{variant_name}__{family}"] = (new_path, "Top")
for family in family_names:
    base_path = _path_for(f"screen_baseline_{family}")
    performance_selections[f"screen__{family}"] = (base_path, "Top")
    first_baseline_path = first_baseline_path or base_path
performance_selections["Benchmark"] = (first_baseline_path, "Bench")

top_curves = combine_backtest_performances(
    export_dir=EXPORT_DIR,
    selections=performance_selections,
)
top_curves.to_csv(EXPORT_DIR / "performance_top_curves.csv", index=True)
top_ratios = calculate_performance_ratios(top_curves, benchmark_column="Benchmark")
top_ratios.to_csv(EXPORT_DIR / "performance_ratios.csv", index=True)

run_manifest = {
    "market": MARKET,
    "benchmark": BENCHMARK,
    "start_date": START_DATE,
    "period_breakpoints": PERIOD_BREAKPOINTS,
    "percentile": PERCENTILE,
    "rebalancing_frequency": 1,
    "fill_method": "copy",
    "incremental_weights": {
        "baseline": INCREMENTAL_BASELINE_WEIGHT,
        "candidate": INCREMENTAL_CANDIDATE_WEIGHT,
    },
    "baseline_columns": BASELINE_COLUMNS,
    "selection_variants": list(SELECTION_VARIANTS),
    "selection_manifest": SELECTION_ROWS,
    "source_evidence_report": str(EVIDENCE_REPORT),
    "gate": {
        "performance": "active_cagr > 0 and top_worst_cagr > 0 and top_information_ratio > 0",
        "strict_comparable": "performance gate and robust_score > 0",
        "short_period_note": "robust_score is diagnostic when robust_score_comparable is false",
    },
    "outputs": [
        "backtest_metrics.csv",
        "backtest_registry.json",
        "family_composite_vs_screen.csv",
        "family_composite_vs_screen_total.csv",
        "incremental_effects.csv",
        "incremental_effects_total.csv",
        "performance_top_curves.csv",
        "performance_ratios.csv",
        "selection_manifest.csv",
        "run_manifest.json",
    ],
}
SELECTION_MANIFEST.to_csv(EXPORT_DIR / "selection_manifest.csv", index=False)
with (EXPORT_DIR / "run_manifest.json").open("w", encoding="utf-8") as handle:
    json.dump(run_manifest, handle, ensure_ascii=False, indent=2)

print(f"Répertoire des résultats : {EXPORT_DIR}")
display(
    family_comparison.loc[
        family_comparison["period_id"].eq("total"),
        [
            "variant",
            "family",
            "active_cagr_new",
            "active_cagr_screen",
            "delta_active_cagr",
            "top_information_ratio_new",
            "top_information_ratio_screen",
            "delta_top_information_ratio",
            "robust_score_new",
            "robust_score_screen",
            "strict_comparable_improvement",
        ],
    ].sort_values("delta_active_cagr", ascending=False)
)
display(
    incremental_effects.loc[
        incremental_effects["period_id"].eq("total"),
        [
            "variant",
            "family",
            "role",
            "variable",
            "dimension",
            "delta_active_cagr",
            "delta_top_worst_cagr",
            "delta_top_information_ratio",
            "delta_robust_score",
            "incremental_perf_improved",
            "incremental_risk_not_worse",
        ],
    ].sort_values(
        ["variant", "family", "delta_active_cagr"], ascending=[True, True, False]
    )
)


## 5. Ordre de lecture après exécution

1. Consulter family_composite_vs_screen_total.csv et comparer la colonne variant entre stable_core, recent_confirmation et current_control.
2. Consulter les lignes par période de family_composite_vs_screen.csv pour vérifier que l'amélioration ne provient pas d'une seule période ou d'un échantillon court.
3. Consulter incremental_effects_total.csv et incremental_effects.csv : une composante ne doit être proposée à l'ajout que si les métriques de performance s'améliorent sans détérioration du risque.
4. Revenir ensuite à backtest_metrics.csv et backtest_registry.json afin de vérifier la composition, les périodes, le benchmark, observation_count et la provenance.

Ce notebook utilise le sous-ensemble Top12 fourni par le rapport d'évidence ; les résultats ne doivent donc pas être interprétés comme une sélection non biaisée de l'univers complet des variables.